# 03 · Train Random Forest on PaySim

Reads `paysim.csv` from the HF dataset repo, fits the project's shared
`PaySimPreprocessor`, oversamples with SMOTE, trains a `RandomForestClassifier`,
evaluates against a rule-based baseline (study research objective 4), and
pushes the model + preprocessor + metrics to the HF model repo.

The preprocessor and rule-based baseline both live in the shipped `src/` package
so the **same** code paths run during training (here) and during inference
(local Streamlit GUI). That keeps joblib unpickling deterministic across
environments.

In [ ]:
import sys, subprocess
for p in ('huggingface_hub','pandas','scikit-learn','imbalanced-learn','joblib','statsmodels'):
    mod = {'scikit-learn':'sklearn','imbalanced-learn':'imblearn'}.get(p, p)
    try: __import__(mod)
    except ImportError: subprocess.check_call([sys.executable,'-m','pip','install','-q',p])
print('deps OK')

In [ ]:
# Make the shipped src/ package importable inside this notebook
import sys
from pathlib import Path
APP_ROOT = Path('/home/user/app')
if str(APP_ROOT) not in sys.path:
    sys.path.insert(0, str(APP_ROOT))

import os, json
import numpy as np
import pandas as pd
import joblib
from huggingface_hub import HfApi, create_repo, hf_hub_download, whoami

from src.data.preprocessor import PaySimPreprocessor
from src.models.rule_based_baseline import paysim_rule_predict

HF_TOKEN = os.environ['HF_TOKEN']
HF_USERNAME = os.environ.get('HF_USERNAME') or whoami(token=HF_TOKEN)['name']
DATASET_REPO = f'{HF_USERNAME}/fraud-detection-datasets'
MODEL_REPO   = f'{HF_USERNAME}/fraud-detection-models'
print('model repo:', MODEL_REPO)

## 1. Load PaySim from the HF dataset repo

In [ ]:
csv = hf_hub_download(repo_id=DATASET_REPO, filename='paysim.csv', repo_type='dataset', token=HF_TOKEN)
df = pd.read_csv(csv)
print(df.shape, 'fraud rate', df.isFraud.mean())

## 2. Preprocess + SMOTE + fit RandomForest

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from imblearn.over_sampling import SMOTE

RANDOM_STATE = 42
TEST_SIZE = 0.2

preprocessor = PaySimPreprocessor()
X, y = preprocessor.fit_transform(df)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=TEST_SIZE, stratify=y, random_state=RANDOM_STATE)

fraud_rate_before = float(y_tr.mean())
if y_tr.sum() > 5:
    sm = SMOTE(random_state=RANDOM_STATE, k_neighbors=min(5, int(y_tr.sum()) - 1))
    X_tr, y_tr = sm.fit_resample(X_tr, y_tr)
fraud_rate_after = float(y_tr.mean())
print(f'fraud rate before SMOTE: {fraud_rate_before:.4%}  after: {fraud_rate_after:.4%}  train rows: {len(y_tr):,}')

rf = RandomForestClassifier(
    n_estimators=150, max_depth=12, min_samples_split=5,
    class_weight='balanced', random_state=RANDOM_STATE, n_jobs=-1,
)
rf.fit(X_tr, y_tr)
print('done')

## 3. Evaluate — RF, rule-based baseline, McNemar

In [ ]:
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, roc_curve, confusion_matrix)
from statsmodels.stats.contingency_tables import mcnemar

def evaluate(y_true, y_pred, y_score=None, label='model'):
    cm = confusion_matrix(y_true, y_pred, labels=[0,1])
    tn, fp, fn, tp = cm.ravel()
    out = {'label': label,
           'accuracy': float(accuracy_score(y_true, y_pred)),
           'precision': float(precision_score(y_true, y_pred, zero_division=0)),
           'recall': float(recall_score(y_true, y_pred, zero_division=0)),
           'f1': float(f1_score(y_true, y_pred, zero_division=0)),
           'true_negatives': int(tn), 'false_positives': int(fp),
           'false_negatives': int(fn), 'true_positives': int(tp)}
    if y_score is not None and len(set(y_true)) > 1:
        out['roc_auc'] = float(roc_auc_score(y_true, y_score))
        fpr, tpr, _ = roc_curve(y_true, y_score)
        out['roc_curve'] = {'fpr': fpr.tolist(), 'tpr': tpr.tolist(), 'auc': out['roc_auc']}
    return out

y_pred = rf.predict(X_te); y_prob = rf.predict_proba(X_te)[:, 1]
rf_metrics = evaluate(y_te, y_pred, y_prob, 'random_forest')

# Apply the rule-based baseline to the same test rows (using untransformed df)
df_idx = df.reset_index(drop=True)
_, idx_te = train_test_split(df_idx.index, test_size=TEST_SIZE, stratify=df.isFraud, random_state=RANDOM_STATE)
rule_pred = paysim_rule_predict(df_idx.loc[idx_te])
rule_metrics = evaluate(y_te, rule_pred, None, 'rule_based_paysim')

rf_right = (y_pred == y_te); base_right = (rule_pred == y_te)
a = int((rf_right & base_right).sum()); b = int((base_right & ~rf_right).sum())
c = int((~base_right & rf_right).sum()); d = int((~rf_right & ~base_right).sum())
mc = mcnemar([[a,b],[c,d]], exact=(b+c)<25, correction=True)
mcnemar_result = {
    'table': {'ml_right_base_right': a,'ml_wrong_base_right': b,'ml_right_base_wrong': c,'ml_wrong_base_wrong': d},
    'statistic': float(mc.statistic) if mc.statistic is not None else None,
    'p_value': float(mc.pvalue),
    'reject_null_at_0.05': bool(mc.pvalue < 0.05),
    'interpretation': ('RF significantly improves accuracy (reject H1, accept HA)'
                       if mc.pvalue < 0.05 and c > b else 'Insufficient evidence to reject H1'),
}
print(json.dumps({'rf': rf_metrics, 'rule_based': rule_metrics, 'mcnemar': mcnemar_result}, indent=2)[:1200])

## 4. Persist + push to HF model repo

In [ ]:
OUT = Path('/tmp/fraudguard_artifacts'); OUT.mkdir(exist_ok=True)
joblib.dump(rf, OUT / 'random_forest.joblib')
joblib.dump(preprocessor, OUT / 'paysim_preprocessor.joblib')

create_repo(repo_id=MODEL_REPO, repo_type='model', token=HF_TOKEN, private=True, exist_ok=True)
existing = {}
try:
    existing = json.loads(Path(hf_hub_download(repo_id=MODEL_REPO, filename='metadata.json', token=HF_TOKEN)).read_text())
except Exception:
    pass
existing['paysim'] = {
    'rf': rf_metrics,
    'rule_based': rule_metrics,
    'mcnemar': mcnemar_result,
    'feature_importances': dict(zip(preprocessor.feature_cols, rf.feature_importances_.tolist())),
    'fraud_rate_before_smote': fraud_rate_before,
    'fraud_rate_after_smote': fraud_rate_after,
    'n_train_after_smote': int(len(y_tr)),
}
(OUT / 'metadata.json').write_text(json.dumps(existing, indent=2, default=str))

api = HfApi()
for f in ('random_forest.joblib','paysim_preprocessor.joblib','metadata.json'):
    print('upload', f)
    api.upload_file(path_or_fileobj=str(OUT / f), path_in_repo=f,
                    repo_id=MODEL_REPO, repo_type='model', token=HF_TOKEN,
                    commit_message=f'Update {f} from RF training')
print('done — view at https://huggingface.co/' + MODEL_REPO)